In [ ]:
import os
print(os.getcwd())
print(os.listdir("."))
print(os.path.exists("IMG_1726.JPG"))

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1) Load model
model_type = "DPT_Large"   # or "DPT_Hybrid", "MiDaS_small"
midas = torch.hub.load("intel-isl/MiDaS", model_type)

# 2) Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
midas.eval()

# 3) Load transforms
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = (
    midas_transforms.dpt_transform
    if model_type in ["DPT_Large", "DPT_Hybrid"]
    else midas_transforms.small_transform
)

# 4) Read image
img_path = "IMG_1726.JPG"

print("Current working dir:", os.getcwd())
print("File exists:", os.path.exists(img_path))

img_bgr = cv2.imread(img_path)

if img_bgr is None:
    raise FileNotFoundError(
        f"Could not read image: {img_path}\n"
        f"Current directory: {os.getcwd()}\n"
        f"Try using the full file path."
    )

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# 5) Preprocess
input_batch = transform(img_rgb).to(device)

# 6) Inference
with torch.no_grad():
    prediction = midas(input_batch)
    prediction = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=img_rgb.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

depth = prediction.cpu().numpy()

# 7) Normalize for display
depth_vis = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

# 8) Show
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(depth_vis, cmap="plasma")
plt.title("MiDaS Depth")
plt.axis("off")

plt.tight_layout()
plt.show()